In [ ]:
import os
import json
import time
import pandas as pd
import fitz  # PyMuPDF
import google.generativeai as genai
from dotenv import load_dotenv

# Carrega variáveis de ambiente (API_KEY guardada em .env)
load_dotenv()
api_key = os.getenv('GENAI_API_KEY')

if not api_key:
    raise RuntimeError('GENAI_API_KEY não encontrada! Crie um arquivo .env com GENAI_API_KEY=sua_chave')

# Configuração da API Key
genai.configure(api_key=api_key)

# Configura o modelo para ser rigoroso e retornar JSON
model = genai.GenerativeModel(
    model_name='gemini-3.1-flash-lite-preview',
    generation_config={
        "response_mime_type": "application/json",
        "temperature": 0.1
    }
)

def extrair_texto_pdf(caminho_pdf):
    """Lê o PDF e retorna o texto completo."""
    texto = ""
    try:
        doc = fitz.open(caminho_pdf)
        for pagina in doc:
            texto += pagina.get_text()
        doc.close()
    except Exception as e:
        print(f"Erro ao ler {caminho_pdf}: {e}")
    return texto

def processar_correcao(gabarito, trabalho_aluno):
    """Envia o texto para o Gemini e retorna o dicionário de correção."""
    prompt = f"""
    Atue como um professor de engenharia de requisitos de software. Sua tarefa é auditar um relato de reunião 
    feito por alunos e compará-lo com a lista de requisitos mestre do cliente.

    ---
    LISTA MESTRE DO CLIENTE (GABARITO):
    {gabarito}
    ---
    TRABALHO DO ALUNO (RELATO DA REUNIÃO):
    {trabalho_aluno}
    ---

    Instruções de Avaliação:
    1. Identifique quais requisitos do gabarito foram mapeados corretamente.
    2. Liste requisitos do gabarito que foram esquecidos pelos alunos.
    3. Identifique se os alunos inventaram funcionalidades que não estavam no gabarito (alucinações de requisitos).
    4. Avalie a clareza da escrita técnica.
    
    Retorne um objeto JSON com:
    "nota": (numérico de 0 a 10),
    "requisitos_identificados": (string com os itens encontrados),
    "requisitos_faltantes": (string com o que faltou),
    "feedback": (texto curto explicando a nota)
    """
    
    try:
        response = model.generate_content(prompt)
        return json.loads(response.text)
    except Exception as e:
        print(f"Erro na API: {e}")
        return None

# --- Execução ---

diretorio_trabalhos = "./grad" # Pasta com os PDFs
arquivo_gabarito = "GabaritoAssembleiaOnLine.pdf"
planilha_final = "Consolidado_Grad_Correcoes_Eng_Requisitos.xlsx"

# 1. Ler Gabarito uma única vez
print("Lendo gabarito...")
texto_gabarito = extrair_texto_pdf(arquivo_gabarito)
print(texto_gabarito[:100])

# 2. Lista para acumular os resultados
resultados_totais = []

# 3. Iterar sobre os arquivos dos alunos
if not os.path.exists(diretorio_trabalhos):
    print(f"Erro: A pasta {diretorio_trabalhos} não existe.")
else:
    arquivos = [f for f in os.listdir(diretorio_trabalhos) if f.endswith('.pdf')]
    print(f"Encontrados {len(arquivos)} trabalhos para corrigir.\n")

    for nome_arquivo in arquivos:
        print(f"Corrigindo: {nome_arquivo}...")
        caminho_completo = os.path.join(diretorio_trabalhos, nome_arquivo)
        
        texto_trabalho = extrair_texto_pdf(caminho_completo)
        
        if texto_trabalho:
            correcao = processar_correcao(texto_gabarito, texto_trabalho)
            
            if correcao:
                # Adiciona o nome do arquivo para identificar o aluno na planilha
                correcao["Arquivo"] = nome_arquivo
                resultados_totais.append(correcao)
            
            # Pequena pausa para evitar limite de taxa da API (Rate Limit)
            time.sleep(2) 

    # 4. Salvar na Planilha
    if resultados_totais:
        df = pd.DataFrame(resultados_totais)
        
        # Reordenar colunas para o nome do arquivo ser a primeira
        cols = ['Arquivo', 'nota', 'requisitos_identificados', 'requisitos_faltantes', 'feedback']
        df = df[cols]
        
        df.to_excel(planilha_final, index=False)
        print(f"\nSucesso! Planilha '{planilha_final}' gerada com {len(resultados_totais)} correções.")
    else:
        print("Nenhum resultado foi processado.")


Lendo gabarito...
1. Gestão de Acessos e Identidade 
• 
Login Único: Acesso exclusivo via e-mail e senha cadastrados. 
Encontrados 12 trabalhos para corrigir.

Corrigindo: 20249022000_DAVI SOUSA.pdf...
Corrigindo: 20249021970_KAUAN FRANCISCO.pdf...
Corrigindo: 20249012050_LUANDERSON OLIVEIRA.pdf...
Corrigindo: 20229024026_ARIELLY CRISTINY.pdf...
Corrigindo: 20249012336_MATHEUS CARNEIRO.pdf...
Corrigindo: 20249011553_SAMARA FEITOSA.pdf...
Corrigindo: 20199041487_ARTHUR RABELO.pdf...
Corrigindo: 20249021433_ANTONIO INACIO.pdf...
Corrigindo: 20249012229_ARTHUR MAGNO.pdf...
Corrigindo: 20239046312_OTAVIO DA.pdf...
Corrigindo: 20239009346_CARLOS EMANUEL.pdf...
Corrigindo: 20229047951_ALLYSON KAWA.pdf...

Sucesso! Planilha 'Consolidado_Grad_Correcoes_Eng_Requisitos.xlsx' gerada com 12 correções.


In [ ]:
print("Iniciando correção...")

Iniciando correção...
